# 22 FS3 Pruning And Redesign Validation

This notebook is the `FS3` decision layer.

It comes after the `FS3` benchmark and staged-ablation notebooks, so it works on top of the frozen `FS2` foundation rather than redefining it.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [ ]:
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    load_fs3_ablation_report_dataset,
    select_feature_family_metric_slice,
    summarize_ablation_cross_model,
)

apply_notebook_display_defaults()

RUN_DIAGNOSTICS = False
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")


_combined_store = load_external_feature_store(config)
_combined_layer2_targets = supported_layer2_target_blocks(_combined_store.experiment_map())
PRUNING_BASELINE_PARENT_SPECS = [{'parent_run_label': 'lear_fs3_combo_promoted_benchmark', 'model_family': 'lear', 'model_label': 'LEAR', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_promoted_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost', 'model_name': 'xgboost_fs3_combo_promoted'}]
PRUNING_BASELINE_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
PRUNING_BASELINE_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _combined_layer2_targets
)

PRUNING_BASELINE_REPORT = load_fs3_ablation_report_dataset(
    output_root,
    config,
    parent_specs=PRUNING_BASELINE_PARENT_SPECS,
    scheme_requests=PRUNING_BASELINE_SCHEME_REQUESTS,
)
PRUNING_BASELINE_AVAILABILITY = PRUNING_BASELINE_REPORT["availability"]
PRUNING_BASELINE_BUNDLES = PRUNING_BASELINE_REPORT["bundles"]
PRUNING_BASELINE_SUMMARY = PRUNING_BASELINE_REPORT["summary"]
PRUNING_BASELINE_BY_ORIGIN = PRUNING_BASELINE_REPORT["by_origin"]


In [ ]:
from IPython.display import Markdown

from hourly_da.core.ablation_blocks import (
    SCHEME_NAME_LAYER_1,
    SCHEME_NAME_LAYER_2,
    SCHEME_NAME_STAGE_A,
    supported_layer2_target_blocks,
)
from hourly_da.notebook_support import (
    ablation_effect_label,
    annotate_ablation_metric_slice,
    apply_notebook_display_defaults,
    load_fs3_ablation_report_dataset,
    select_feature_family_metric_slice,
    summarize_ablation_cross_model,
)

apply_notebook_display_defaults()

RUN_DIAGNOSTICS = False
PRIMARY_SPLIT = "validation"
PRIMARY_METRIC = "mae"
VISIBLE_REPORTING_LEVELS = ("d_only", "stitched_all_horizon")


_combined_store = load_external_feature_store(config)
_combined_layer2_targets = supported_layer2_target_blocks(_combined_store.experiment_map())
PRUNING_CANDIDATE_PARENT_SPECS = [{'parent_run_label': 'lear_fs3_combo_pruned_candidate_benchmark', 'model_family': 'lear', 'model_label': 'LEAR candidate', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_pruned_candidate_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost candidate', 'model_name': 'xgboost_fs3_combo_promoted'}]
PRUNING_CANDIDATE_SCHEME_REQUESTS = [
    {"scheme_name": SCHEME_NAME_STAGE_A, "target_block": None},
    {"scheme_name": SCHEME_NAME_LAYER_1, "target_block": None},
]
PRUNING_CANDIDATE_SCHEME_REQUESTS.extend(
    {"scheme_name": SCHEME_NAME_LAYER_2, "target_block": target_block}
    for target_block in _combined_layer2_targets
)

PRUNING_CANDIDATE_REPORT = load_fs3_ablation_report_dataset(
    output_root,
    config,
    parent_specs=PRUNING_CANDIDATE_PARENT_SPECS,
    scheme_requests=PRUNING_CANDIDATE_SCHEME_REQUESTS,
)
PRUNING_CANDIDATE_AVAILABILITY = PRUNING_CANDIDATE_REPORT["availability"]
PRUNING_CANDIDATE_BUNDLES = PRUNING_CANDIDATE_REPORT["bundles"]
PRUNING_CANDIDATE_SUMMARY = PRUNING_CANDIDATE_REPORT["summary"]
PRUNING_CANDIDATE_BY_ORIGIN = PRUNING_CANDIDATE_REPORT["by_origin"]


## Optional execution hook

Use this notebook to validate whether a suspicious `FS3` block should actually be removed or redesigned.

Recommended sequence:
1. choose one small candidate block set per model from the baseline Layer 1 results
2. rerun the revised parent benchmark
3. inspect the parent metric delta
4. rerun staged ablation on the candidate lineage only if the revised parent itself looks promising


In [ ]:
ALLOW_HEAVY_RERUN = False
RUN_CANDIDATE_PARENT_BENCHMARKS = False
RUN_CANDIDATE_ABLATION = False
REFRESH_CANDIDATE_COMPARISON = None
RUN_STAGE_A = True
RUN_LAYER_1 = True
RUN_LAYER_2 = False

LAYER_2_TARGET_BLOCKS = {'domestic_day_ahead_fundamentals': False, 'neighbor_only_day_ahead_fundamentals': False, 'domestic_historical_fundamentals': False, 'neighbor_only_historical_fundamentals': False, 'engineered_endogenous_history_stats': False, 'weekly_same_hour_lag_structure': False, 'structural_capacity': False}
CANDIDATE_BLOCK_SELECTION = {
    "lear": [],
    "xgboost": [],
}

BASELINE_PARENT_SPECS = [{'parent_run_label': 'lear_fs3_combo_promoted_benchmark', 'model_family': 'lear', 'model_label': 'LEAR', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_promoted_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost', 'model_name': 'xgboost_fs3_combo_promoted'}]
CANDIDATE_PARENT_SPECS = [{'parent_run_label': 'lear_fs3_combo_pruned_candidate_benchmark', 'model_family': 'lear', 'model_label': 'LEAR candidate', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_pruned_candidate_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost candidate', 'model_name': 'xgboost_fs3_combo_promoted'}]
selected_layer2_targets = [
    block_name
    for block_name, enabled in LAYER_2_TARGET_BLOCKS.items()
    if bool(enabled)
]

if ALLOW_HEAVY_RERUN:
    if RUN_CANDIDATE_PARENT_BENCHMARKS:
        for baseline_spec, candidate_spec in zip(BASELINE_PARENT_SPECS, CANDIDATE_PARENT_SPECS):
            model_family = str(baseline_spec["model_family"])
            excluded_blocks = [str(value) for value in CANDIDATE_BLOCK_SELECTION.get(model_family, []) if str(value).strip()]
            if not excluded_blocks:
                print(f"Skipping {model_family} candidate parent rerun because no blocks were selected.")
                continue
            command = [
                sys.executable,
                str(PACKAGE_ROOT / "run_revised_parent_benchmark.py"),
                "--fs-level",
                "FS3",
                "--model-family",
                model_family,
                "--parent-run-label",
                str(baseline_spec["parent_run_label"]),
                "--run-label",
                str(candidate_spec["parent_run_label"]),
                "--ablation-scheme",
                "layer1_mutually_exclusive",
                "--excluded-blocks",
                *excluded_blocks,
            ]
            run_command_with_live_output(command)

    if RUN_CANDIDATE_ABLATION:
        ablation_schemes = []
        if RUN_STAGE_A:
            ablation_schemes.append("stage_a_top_level")
        if RUN_LAYER_1:
            ablation_schemes.append("layer1_mutually_exclusive")
        if RUN_LAYER_2 and selected_layer2_targets:
            ablation_schemes.append("layer2_subgroups")
        if not ablation_schemes:
            raise RuntimeError("At least one staged ablation layer must be enabled for the candidate rerun.")

        for candidate_spec in CANDIDATE_PARENT_SPECS:
            model_family = str(candidate_spec["model_family"])
            excluded_blocks = [str(value) for value in CANDIDATE_BLOCK_SELECTION.get(model_family, []) if str(value).strip()]
            if not excluded_blocks:
                print(f"Skipping {model_family} candidate ablation because no candidate block set was selected.")
                continue
            command = [
                sys.executable,
                str(PACKAGE_ROOT / "run_staged_block_ablation.py"),
                "--fs-level",
                "FS3",
                "--model-family",
                model_family,
                "--parent-run-label",
                str(candidate_spec["parent_run_label"]),
                "--ablation-schemes",
                *ablation_schemes,
            ]
            if RUN_LAYER_2 and selected_layer2_targets:
                command.extend(["--layer2-target-block", *selected_layer2_targets])
            run_command_with_live_output(command)

    if False and bool(REFRESH_CANDIDATE_COMPARISON):
        def _active_run_label(model_family: str) -> str:
            selected_blocks = [str(value) for value in CANDIDATE_BLOCK_SELECTION.get(model_family, []) if str(value).strip()]
            if selected_blocks:
                matching = [spec for spec in CANDIDATE_PARENT_SPECS if str(spec["model_family"]) == model_family]
                if matching:
                    return str(matching[0]["parent_run_label"])
            matching = [spec for spec in BASELINE_PARENT_SPECS if str(spec["model_family"]) == model_family]
            if not matching:
                raise RuntimeError(f"No baseline spec was found for model family {model_family}.")
            return str(matching[0]["parent_run_label"])

        command = [
            sys.executable,
            str(PACKAGE_ROOT / "run_model_comparison.py"),
            "--fs-level",
            "FS3",
            "--run-label",
            "",
            "--lear-run-label",
            _active_run_label("lear"),
            "--xgboost-run-label",
            _active_run_label("xgboost"),
            "--prophet-run-label",
            "prophet_benchmark",
        ]
        run_command_with_live_output(command)
else:
    print("Candidate reruns are disabled. Set ALLOW_HEAVY_RERUN = True when you are ready to validate a pruning or redesign choice.")
    print(f"RUN_CANDIDATE_PARENT_BENCHMARKS={RUN_CANDIDATE_PARENT_BENCHMARKS}")
    print(f"RUN_CANDIDATE_ABLATION={RUN_CANDIDATE_ABLATION}")
    if False:
        print(f"REFRESH_CANDIDATE_COMPARISON={REFRESH_CANDIDATE_COMPARISON}")
    print(f"Selected candidate blocks={CANDIDATE_BLOCK_SELECTION}")
    print(f"Layer 2 targets={selected_layer2_targets or 'none'}")


## Baseline Availability


In [ ]:
availability_view = PRUNING_BASELINE_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

display(
    Markdown(
        "Active synthesis scope for baseline FS3 lineage: only compatible staged-ablation bundles are included below. "
        "Saved aggregates with stale scheme hashes, stale taxonomy hashes, or invalid stored preflight are excluded."
    )
)


## Suggested Candidate Actions


In [ ]:
def _action_from_buckets(d_bucket: str, stitched_bucket: str) -> str:
    harmful = {"strong_harmful", "mild_harmful"}
    helpful = {"strong_helpful", "mild_helpful"}
    neutral = {"near_zero_uncertain", "insufficient_data"}
    if d_bucket in harmful and stitched_bucket in harmful:
        return "prune_candidate"
    if (d_bucket in harmful and stitched_bucket in neutral) or (stitched_bucket in harmful and d_bucket in neutral):
        return "prune_candidate"
    if (d_bucket in harmful and stitched_bucket in helpful) or (d_bucket in helpful and stitched_bucket in harmful):
        return "redesign_candidate"
    if d_bucket in neutral and stitched_bucket in neutral:
        return "ambiguous"
    return "keep_or_monitor"


summary_scope = PRUNING_BASELINE_SUMMARY.copy()
layer1_scope = summary_scope[summary_scope["scheme_name"].astype(str) == "layer1_mutually_exclusive"].copy()

if layer1_scope.empty:
    print("No compatible Layer 1 summary exists yet for the baseline lineage.")
else:
    action_rows = []
    for model_family in sorted(layer1_scope["model_family"].astype(str).unique()):
        d_only = annotate_ablation_metric_slice(
            select_feature_family_metric_slice(
                layer1_scope[layer1_scope["model_family"].astype(str) == model_family],
                split="validation",
                reporting_level="d_only",
                metric="mae",
            )
        )[["feature_family", "effect_bucket", "relative_delta"]].rename(
            columns={
                "effect_bucket": "d_only_bucket",
                "relative_delta": "d_only_relative_delta",
            }
        )
        stitched = annotate_ablation_metric_slice(
            select_feature_family_metric_slice(
                layer1_scope[layer1_scope["model_family"].astype(str) == model_family],
                split="validation",
                reporting_level="stitched_all_horizon",
                metric="mae",
            )
        )[["feature_family", "effect_bucket", "relative_delta"]].rename(
            columns={
                "effect_bucket": "stitched_bucket",
                "relative_delta": "stitched_relative_delta",
            }
        )
        merged = d_only.merge(stitched, on="feature_family", how="outer")
        if merged.empty:
            continue
        merged["model_family"] = model_family
        merged["suggested_action"] = merged.apply(
            lambda row: _action_from_buckets(
                str(row.get("d_only_bucket", "insufficient_data")),
                str(row.get("stitched_bucket", "insufficient_data")),
            ),
            axis=1,
        )
        action_rows.append(merged)

    if not action_rows:
        print("No candidate action rows could be derived from the baseline summary.")
    else:
        action_table = pd.concat(action_rows, ignore_index=True)
        display(
            action_table[
                [
                    "model_family",
                    "feature_family",
                    "d_only_bucket",
                    "d_only_relative_delta",
                    "stitched_bucket",
                    "stitched_relative_delta",
                    "suggested_action",
                ]
            ]
            .rename(
                columns={
                    "model_family": "Model family",
                    "feature_family": "Block",
                    "d_only_bucket": "D-only effect",
                    "d_only_relative_delta": "D-only relative delta",
                    "stitched_bucket": "Stitched effect",
                    "stitched_relative_delta": "Stitched relative delta",
                    "suggested_action": "Suggested action",
                }
            )
            .style
            .format(
                {
                    "D-only relative delta": "{:+.3%}",
                    "Stitched relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )


## Candidate Benchmark Delta


In [ ]:
baseline_specs = [{'parent_run_label': 'lear_fs3_combo_promoted_benchmark', 'model_family': 'lear', 'model_label': 'LEAR', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_promoted_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost', 'model_name': 'xgboost_fs3_combo_promoted'}]
candidate_specs = [{'parent_run_label': 'lear_fs3_combo_pruned_candidate_benchmark', 'model_family': 'lear', 'model_label': 'LEAR candidate', 'model_name': 'lear_fs3_combo_promoted'}, {'parent_run_label': 'xgboost_fs3_combo_pruned_candidate_benchmark', 'model_family': 'xgboost', 'model_label': 'XGBoost candidate', 'model_name': 'xgboost_fs3_combo_promoted'}]
comparison_rows = []

for baseline_spec, candidate_spec in zip(baseline_specs, candidate_specs):
    baseline_run_dir = latest_run_or_none(str(baseline_spec["parent_run_label"]))
    candidate_run_dir = latest_run_or_none(str(candidate_spec["parent_run_label"]))
    if baseline_run_dir is None:
        continue
    baseline_metrics = load_csv(baseline_run_dir, "metrics_by_reporting_level.csv")
    baseline_metrics = baseline_metrics[baseline_metrics["model"].astype(str) == str(baseline_spec["model_name"])].copy()
    if baseline_metrics.empty:
        continue
    candidate_metrics = baseline_metrics.iloc[0:0].copy()
    if candidate_run_dir is not None:
        candidate_metrics = load_csv(candidate_run_dir, "metrics_by_reporting_level.csv")
        candidate_metrics = candidate_metrics[candidate_metrics["model"].astype(str) == str(candidate_spec["model_name"])].copy()

    for split_name in ("validation", "test"):
        for reporting_level in ("d_only", "stitched_all_horizon"):
            base_row = baseline_metrics[
                (baseline_metrics["dataset_split"].astype(str) == split_name)
                & (baseline_metrics["reporting_level"].astype(str) == reporting_level)
            ].head(1)
            candidate_row = candidate_metrics[
                (candidate_metrics["dataset_split"].astype(str) == split_name)
                & (candidate_metrics["reporting_level"].astype(str) == reporting_level)
            ].head(1)
            comparison_rows.append(
                {
                    "Model family": baseline_spec["model_family"],
                    "Model label": baseline_spec["model_label"],
                    "Dataset split": split_name,
                    "Reporting level": reporting_level,
                    "Baseline run": baseline_spec["parent_run_label"],
                    "Candidate run": candidate_spec["parent_run_label"] if candidate_run_dir is not None else "",
                    "Baseline MAE": float(base_row["mae"].iloc[0]) if not base_row.empty else float("nan"),
                    "Candidate MAE": float(candidate_row["mae"].iloc[0]) if not candidate_row.empty else float("nan"),
                    "MAE delta (candidate - baseline)": (
                        float(candidate_row["mae"].iloc[0]) - float(base_row["mae"].iloc[0])
                        if (not base_row.empty and not candidate_row.empty)
                        else float("nan")
                    ),
                    "Baseline rMAE": float(base_row["rmae_vs_official_naive"].iloc[0]) if not base_row.empty else float("nan"),
                    "Candidate rMAE": float(candidate_row["rmae_vs_official_naive"].iloc[0]) if not candidate_row.empty else float("nan"),
                }
            )

comparison_frame = pd.DataFrame(comparison_rows)
if comparison_frame.empty:
    print("No baseline benchmark rows were available for the candidate comparison.")
else:
    display(
        comparison_frame.style
        .format(
            {
                "Baseline MAE": "{:.4f}",
                "Candidate MAE": "{:.4f}",
                "MAE delta (candidate - baseline)": "{:+.4f}",
                "Baseline rMAE": "{:.4f}",
                "Candidate rMAE": "{:.4f}",
            }
        )
        .hide(axis="index")
    )


## Candidate Ablation Availability


In [ ]:
availability_view = PRUNING_CANDIDATE_AVAILABILITY[
    [
        "model_label",
        "scheme_name",
        "layer_name",
        "target_block",
        "availability_status",
        "latest_timestamp_label",
        "status_note",
    ]
].rename(
    columns={
        "model_label": "Model",
        "scheme_name": "Scheme",
        "layer_name": "Layer",
        "target_block": "Layer 2 target",
        "availability_status": "Status",
        "latest_timestamp_label": "Latest timestamp",
        "status_note": "Status note",
    }
)
display(availability_view.style.hide(axis="index"))

display(
    Markdown(
        "Active synthesis scope for candidate FS3 lineage: only compatible staged-ablation bundles are included below. "
        "Saved aggregates with stale scheme hashes, stale taxonomy hashes, or invalid stored preflight are excluded."
    )
)


## Candidate Layer 1 Across Both Models


In [ ]:
scheme_summary = PRUNING_CANDIDATE_SUMMARY[
    PRUNING_CANDIDATE_SUMMARY["scheme_name"].astype(str) == 'layer1_mutually_exclusive'
].copy()

if scheme_summary.empty:
    print("No compatible saved results are available for this layer yet.")
else:
    for reporting_level in VISIBLE_REPORTING_LEVELS:
        metric_slice = select_feature_family_metric_slice(
            scheme_summary,
            split=PRIMARY_SPLIT,
            reporting_level=reporting_level,
            metric=PRIMARY_METRIC,
        )
        display(Markdown(f"### Validation / {reporting_level.replace('_', ' ').title()}"))
        if metric_slice.empty:
            print("No rows are available for this reporting lens.")
            continue

        annotated = annotate_ablation_metric_slice(metric_slice)
        model_view = annotated[
            [
                "model_family",
                "feature_family",
                "parent_value",
                "child_value",
                "delta",
                "relative_delta",
                "effect_label",
            ]
        ].rename(
            columns={
                "model_family": "Model family",
                "feature_family": "Block",
                "parent_value": "Parent value",
                "child_value": "Child value",
                "delta": "Delta",
                "relative_delta": "Relative delta",
                "effect_label": "Interpretation",
            }
        )
        display(
            model_view.style
            .format(
                {
                    "Parent value": "{:.4f}",
                    "Child value": "{:.4f}",
                    "Delta": "{:+.4f}",
                    "Relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )

        combined = summarize_ablation_cross_model(metric_slice)
        display(
            combined.rename(
                columns={
                    "feature_family": "Block",
                    "models_available": "Models available",
                    "model_count": "Model count",
                    "mean_delta": "Mean delta",
                    "mean_relative_delta": "Mean relative delta",
                    "combined_label": "Combined classification",
                }
            ).style
            .format(
                {
                    "Mean delta": "{:+.4f}",
                    "Mean relative delta": "{:+.3%}",
                }
            )
            .hide(axis="index")
        )
